# Notebook 15: V2 Model Comparison (Brier + Reliability Diagrams)

Apples-to-apples Brier score comparison of 4 groups on identical 2025 games:
- **v2 SP_ENHANCED** (3 models: LR, RF, XGB with 17 pruned SP features)
- **v2 TEAM_ONLY** (3 models: LR, RF, XGB with 9 team features)
- **v1** (3 models: LR, RF, XGB with core features)
- **Kalshi market** (opening prices as implied probability)

Plus reliability diagrams for all 6 v2 model/feature-set combinations.

**MDL-03:** Comparison table on identical game sets. **MDL-04:** Reliability diagrams with visual inspection.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

from src.data.kalshi import fetch_kalshi_markets, fetch_kalshi_open_prices
from src.models.evaluate import get_calibration_data

## Cell 1: Load All Prediction Sources

In [ ]:
# V2 predictions (6 models: 3 model types x 2 feature sets)
preds_v2 = pd.read_parquet("../data/results/predictions_2025_v2.parquet")
print(f"V2 predictions: {preds_v2.shape[0]} rows, combos: {preds_v2.groupby(['model_name','feature_set']).size().to_dict()}")

# V1 predictions (3 models, core features)
preds_v1 = pd.read_parquet("../data/results/predictions_2025.parquet")
print(f"V1 predictions: {preds_v1.shape[0]} rows")

# Kalshi opening prices
kalshi_df = fetch_kalshi_markets(max_age_hours=float("inf"))
kalshi_df = fetch_kalshi_open_prices(kalshi_df)

n_open = kalshi_df["kalshi_open_price"].notna().sum()
n_fallback = kalshi_df["kalshi_open_price"].isna().sum()
print(f"Kalshi: {len(kalshi_df)} games ({n_open} open price, {n_fallback} fallback)")
kalshi_df["kalshi_open_price"] = kalshi_df["kalshi_open_price"].fillna(kalshi_df["kalshi_yes_price"])
kalshi_df["game_date"] = pd.to_datetime(kalshi_df["date"])

## Cell 2: Find Common Game Set (Intersection)

In [ ]:
join_cols = ["game_date", "home_team", "away_team"]

# Deduplicate doubleheaders (same date+teams, keep first)
preds_v2_deduped = preds_v2.drop_duplicates(subset=join_cols + ["model_name", "feature_set"], keep="first")
preds_v1_deduped = preds_v1.drop_duplicates(subset=join_cols + ["model_name"], keep="first")
kalshi_deduped = kalshi_df.drop_duplicates(subset=join_cols, keep="first")

# Common intersection via inner join
v2_unique = preds_v2_deduped[join_cols].drop_duplicates()
v1_unique = preds_v1_deduped[join_cols].drop_duplicates()
kalshi_unique = kalshi_deduped[join_cols].drop_duplicates()

common = v2_unique.merge(v1_unique, on=join_cols).merge(kalshi_unique, on=join_cols)
print(f"Common games: {len(common)} (v2: {len(v2_unique)}, v1: {len(v1_unique)}, kalshi: {len(kalshi_unique)})")

## Cell 3: Compute Brier Scores for Each Group

In [ ]:
# Filter each source to common games
v2_sp = preds_v2_deduped[preds_v2_deduped["feature_set"] == "sp_enhanced"].merge(common, on=join_cols)
v2_team = preds_v2_deduped[preds_v2_deduped["feature_set"] == "team_only"].merge(common, on=join_cols)
v1_core = preds_v1_deduped.merge(common, on=join_cols)
kalshi_common = kalshi_deduped.merge(common, on=join_cols)

comparison_rows = []

# V2 SP_ENHANCED - per model
for model in ["lr", "rf", "xgb"]:
    subset = v2_sp[v2_sp["model_name"] == model]
    brier = brier_score_loss(subset["home_win"], subset["prob_calibrated"])
    comparison_rows.append({"group": "v2_sp_enhanced", "model_name": model, "brier_score": round(brier, 4), "n_games": len(subset)})

# V2 TEAM_ONLY - per model
for model in ["lr", "rf", "xgb"]:
    subset = v2_team[v2_team["model_name"] == model]
    brier = brier_score_loss(subset["home_win"], subset["prob_calibrated"])
    comparison_rows.append({"group": "v2_team_only", "model_name": model, "brier_score": round(brier, 4), "n_games": len(subset)})

# V1 - per model
for model in ["lr", "rf", "xgb"]:
    subset = v1_core[v1_core["model_name"] == model]
    brier = brier_score_loss(subset["home_win"], subset["prob_calibrated"])
    comparison_rows.append({"group": "v1", "model_name": model, "brier_score": round(brier, 4), "n_games": len(subset)})

# Kalshi market
outcomes = v2_sp[join_cols + ["home_win"]].drop_duplicates(subset=join_cols)
kalshi_with_outcomes = kalshi_common.merge(outcomes, on=join_cols)
kalshi_brier = brier_score_loss(kalshi_with_outcomes["home_win"], kalshi_with_outcomes["kalshi_open_price"])
comparison_rows.append({"group": "kalshi", "model_name": "market", "brier_score": round(kalshi_brier, 4), "n_games": len(kalshi_with_outcomes)})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("../data/results/brier_comparison.csv", index=False)
print("\nBrier Score Comparison (2025, identical game set):")
print(comparison_df.to_string(index=False))

## Cell 4: Summary Comparison Table (Best Model per Group)

In [ ]:
summary = comparison_df.loc[comparison_df.groupby("group")["brier_score"].idxmin()]
print("Best Model per Group:")
print(summary[["group", "model_name", "brier_score", "n_games"]].to_string(index=False))

## Cell 5: Reliability Diagrams - V2 TEAM_ONLY

In [ ]:
backtest_v2 = pd.read_parquet("../data/results/backtest_results_v2.parquet")

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
for model in ["lr", "rf", "xgb"]:
    frac, mean_pred = get_calibration_data(backtest_v2, model, "team_only", n_bins=10)
    ax.plot(mean_pred, frac, "s-", label=f"{model} (team_only)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability Diagram - v2 TEAM_ONLY")
ax.legend()
plt.tight_layout()
plt.savefig("../data/results/reliability_team_only.png", dpi=150)
plt.show()

## Cell 6: Reliability Diagrams - V2 SP_ENHANCED

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
for model in ["lr", "rf", "xgb"]:
    frac, mean_pred = get_calibration_data(backtest_v2, model, "sp_enhanced", n_bins=10)
    ax.plot(mean_pred, frac, "s-", label=f"{model} (sp_enhanced)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability Diagram - v2 SP_ENHANCED")
ax.legend()
plt.tight_layout()
plt.savefig("../data/results/reliability_sp_enhanced.png", dpi=150)
plt.show()

## Cell 7: Reliability Diagram Assessment

In [ ]:
print("RELIABILITY ASSESSMENT:")
print("=" * 60)
for fs in ["team_only", "sp_enhanced"]:
    print(f"\n{fs.upper()}:")
    for model in ["lr", "rf", "xgb"]:
        frac, mean_pred = get_calibration_data(backtest_v2, model, fs, n_bins=10)
        max_deviation = np.max(np.abs(frac - mean_pred))
        mean_deviation = np.mean(np.abs(frac - mean_pred))
        print(f"  {model}: max_deviation={max_deviation:.4f}, mean_deviation={mean_deviation:.4f}")